# RAG Ingest and Query

Ingest local docs into the RAG vector store and query them in-process.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
for cand in [repo_root, *repo_root.parents]:
    if (cand / 'app').exists():
        repo_root = cand
        break
sys.path.insert(0, str(repo_root))
print('Repo root:', repo_root)


## Data requirements

Place .md or .txt files in data/docs/. The ingest step builds embeddings under data/embeddings/.


## Document inventory


In [ ]:
from pathlib import Path

docs_dir = repo_root / 'data' / 'docs'
print('Docs dir:', docs_dir, 'exists=', docs_dir.exists())
if docs_dir.exists():
    docs = [p for p in docs_dir.rglob('*') if p.suffix.lower() in {'.md', '.txt'}]
    print('Doc count:', len(docs))
    for p in docs[:5]:
        print('-', p.name)


## Links to code


- RAG ingest: `app/rag/ingest.py`


- Retriever: `app/rag/retriever.py`


- API endpoints: `POST /api/rag/ingest`, `POST /api/rag/upload`


In [ ]:
# Update paths if your data lives elsewhere.



In [ ]:
docs_dir = repo_root / 'data' / 'docs'
print('Docs dir exists:', docs_dir.exists(), docs_dir)


## Ingest documents


In [ ]:
from app.rag.ingest import ingest_all_docs
chunks = ingest_all_docs()
print('Chunks ingested:', chunks)


## Retrieve context for a query


In [ ]:
from app.rag.retriever import retrieve_context
query = 'Summarize fraud detection signals.'
results = retrieve_context(query, top_k=3)
for idx, hit in enumerate(results, start=1):
    print(f'[{idx}]', hit.get('source'), hit.get('text', '')[:200])


## Visuals and metrics


In [ ]:
import matplotlib.pyplot as plt
from app.rag.retriever import retrieve_context

query = 'Summarize fraud detection signals.'
results = retrieve_context(query, top_k=5)
if not results:
    print('No RAG results found')
else:
    labels = [r.get('source') or 'chunk-{}'.format(i) for i, r in enumerate(results)]
    scores = [r.get('score', 0.0) for r in results]
    plt.figure(figsize=(6, 3))
    plt.barh(labels, scores)
    plt.title('RAG similarity scores')
    plt.xlabel('score')
    plt.tight_layout()
    plt.show()
    for r in results:
        print(r.get('source'), 'score=', r.get('score'))
